# Ratanabá Guardians — Lux3D + Google Colab

Este notebook usa **o código público do ComfyUI-Lux3D** para preparar a geração de personagens 3D do jogo. A extensão possui licença MIT, mas a **API da Lux3D exige credenciais próprias e pode cobrar por geração**.

**Segurança:** nenhuma geração é enviada com as opções iniciais. O orçamento de 60 créditos do Tripo não autoriza cobranças Lux3D. Não escreva sua chave no notebook, no GitHub ou no chat: use apenas **Colab → Secrets → LUX3D_API_KEY_INTL**.

Ao gerar, os GLBs ficam no diretório de candidatos, sem substituir os modelos atuais do Godot. O BlendRelay MCP funciona separadamente com Blender 4.5+ aberto no Windows.

In [ ]:
# 1. Preparar os dois repositórios. Somente downloads e dependências públicas.
import os, pathlib, subprocess, sys
ROOT = pathlib.Path("/content/ratanaba_guardians")
UPSTREAM = ROOT / "third_party" / "ComfyUI-Lux3D"
if not (ROOT / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch",
        "feature/guardians-v0.2-asset-pipeline",
        "https://github.com/mizaelcosta1/codex.git", str(ROOT)], check=True)
UPSTREAM.parent.mkdir(parents=True, exist_ok=True)
if not (UPSTREAM / ".git").exists():
    subprocess.run(["git","clone","--depth","1",
        "https://github.com/manycore-research/ComfyUI-Lux3D.git",str(UPSTREAM)],check=True)
subprocess.run([sys.executable,"-m","pip","-q","install","requests>=2.25.0"],check=True)
print("Repositórios preparados. Nenhum job Lux3D criado.")

In [ ]:
# 2. Visualizar a especificação e o prompt sem gastar créditos.
import json
SPEC = json.loads((ROOT/"art_source/lux3d_characters.json").read_text(encoding="utf-8"))
CHARACTER = "banido"  # opções: banido, curupira, boitata, mula_sem_cabeca, boto, matinta_pereira
assert CHARACTER in SPEC["characters"]
DESCRIPTION = SPEC["characters"][CHARACTER]["prompt"]
REFERENCE_URL = ""  # opcional: URL HTTP(S) pública da imagem de referência
payload = {
    "prompt": DESCRIPTION,
    "style": "photorealistic",
    "version": "G1-Turbo",
    "faceCount": SPEC["target_faces_for_source"],
    "outputFormat": ["glb","zip"],
    "enablePbr": True,
    "aiPredictSize": True
}
if REFERENCE_URL:
    from urllib.parse import urlsplit
    u = urlsplit(REFERENCE_URL)
    assert u.scheme in ("http","https") and u.netloc
    payload["img"] = REFERENCE_URL
print(json.dumps({"character":CHARACTER,"model":payload["version"],
                  "source_faces_target":payload["faceCount"],
                  "PBR":payload["enablePbr"],"prompt":DESCRIPTION},ensure_ascii=False,indent=2))

In [ ]:
# 3. Carregar a biblioteca de cliente do próprio repositório Lux3D (sem executar ComfyUI).
# A importação do pacote raiz depende do servidor gráfico ComfyUI; aqui importamos apenas o cliente OpenAPI.
import types, sys
pkg = types.ModuleType("lux3d_openapi")
pkg.__path__ = [str(UPSTREAM/"lux3d_openapi")]
sys.modules["lux3d_openapi"] = pkg
from lux3d_openapi.client import Lux3DOpenAPIClient
from lux3d_openapi.contracts import parse_create_task_id
from lux3d_openapi.result_outputs import map_generation_outputs
from lux3d_openapi.task_polling import wait_for_task_result
print("Cliente OpenAPI carregado; geração ainda NÃO solicitada.")

In [ ]:
# 4. Autenticação opcional via Secrets. Nunca imprime o valor da chave.
from google.colab import userdata
try:
    _secret = userdata.get("LUX3D_API_KEY_INTL")
except Exception:
    _secret = None
CLIENT = Lux3DOpenAPIClient(_secret,region="intl",base_api_path="https://api.aholo.ai") if _secret else None
print("Chave configurada:", bool(CLIENT))
print("Antes de gerar, confira preços, termos e saldo em sua conta Lux3D.")

In [ ]:
# 5. ENVIO OPCIONAL, BLOQUEADO POR PADRÃO.
# A API Lux3D pode cobrar. Ative SOMENTE com autorização explícita separada.
APPROVE_PAID_LUX3D_REQUEST = False
TASK_ID = ""
if APPROVE_PAID_LUX3D_REQUEST:
    assert CLIENT is not None, "Configure LUX3D_API_KEY_INTL em Colab Secrets"
    response = CLIENT.create_text_to_3d_task(payload)
    TASK_ID = parse_create_task_id(response)
    print("Job Lux3D submetido:",TASK_ID,"— personagem:",CHARACTER)
else:
    print("TRAVA ATIVA: nenhum job Lux3D criado; nenhum gasto desta célula.")

In [ ]:
# 6. Consulta e download de um JOB EXISTENTE. Não cria geração adicional.
# Preencha TASK_ID de um job já autorizado caso reinicie o notebook.
import hashlib, struct, urllib.request
STAGING = ROOT/"incoming"/"lux3d"
STAGING.mkdir(parents=True,exist_ok=True)
if TASK_ID:
    assert CLIENT is not None
    response, urls = wait_for_task_result(CLIENT,TASK_ID,poll_timeout=900)
    lux3d_zip, glb_url, ply_url = map_generation_outputs(urls)
    assert glb_url, "Job sem saída GLB"
    with urllib.request.urlopen(glb_url,timeout=120) as stream:
        blob = stream.read(120_000_001)
    assert len(blob)<120_000_001, "GLB excedeu 120 MB"
    magic, version, declared = struct.unpack_from("<4sII",blob,0)
    assert magic==b"glTF" and version==2 and declared==len(blob), "Cabeçalho GLB inválido"
    chunk_len,kind=struct.unpack_from("<I4s",blob,12)
    assert kind==b"JSON" and chunk_len%4==0
    content=json.loads(blob[20:20+chunk_len].decode("utf-8"))
    assert content.get("meshes") and content.get("materials"),"Malha/material ausente"
    destination=STAGING/f"{CHARACTER}_lux3d_{TASK_ID}.glb"
    assert not destination.exists(),"Arquivo já existe; não sobrescrever"
    destination.write_bytes(blob)
    audit={"character":CHARACTER,"task_id":TASK_ID,
           "bytes":len(blob),"sha256":hashlib.sha256(blob).hexdigest(),
           "meshes":len(content["meshes"]),"materials":len(content["materials"]),
           "texture_images":len(content.get("images",[])),
           "skins":len(content.get("skins",[])),
           "animations":len(content.get("animations",[]))}
    (STAGING/f"{CHARACTER}_lux3d_{TASK_ID}_audit.json").write_text(
        json.dumps(audit,indent=2),encoding="utf-8")
    print("Candidato salvo:",destination)
    print(json.dumps(audit,indent=2))
else:
    print("Insira um TASK_ID válido quando houver uma geração concluída.")

## Próximo estágio: BlendRelay no Windows

O arquivo gerado deve ser baixado do Colab para o Windows. Em **Blender 4.5+**, execute o setup local do projeto em `tools/Setup-BlendRelay-Windows.ps1`, habilite o add-on BlendRelay e conecte-o ao Codex CLI.

Peça ao agente para importar o GLB em uma coleção `SOURCE_LOCKED`, inspecionar topologia, aplicar retopologia e bake PBR para o Android, criar LODs, validar rig/animações e exportar um *candidato*, nunca sobrescrevendo os assets jogáveis antes dos testes.

**Importante:** o pacote fornecido no Colab não garante rigging, animações, fotorrealismo ou licença comercial. Esses requisitos são validados separadamente.